# CCC Batch Full Run

Run full CCC scenarios through all steps and Vorblatt/Begründung export. Keep credentials in environment variables or the repo-root `.env`, not in notebook output. The runner loads `.env` automatically without overriding values already set in the shell.

A batch is a matrix:

`law_pairs × models × deep_research_modes × repetitions`

Example: 4 law pairs × 2 models × 2 DR modes × 1 repetition = 16 full sessions.

Important config fields:

- `law_pairs`: tuple of built-in law-pair names without `_gueltig.txt` / `_vorschlag.txt`. Empty tuple means all discovered built-in pairs.
- `models`: tuple of `ModelSpec(provider, model)` entries.
- `deep_research_modes`: `(True, False)` runs each scenario once with DR and once without DR.
- `repetitions`: repeat count per law/model/DR combination. Use `1` for exactly one run per combination.
- `concurrency`: how many full sessions may run in parallel. Use `1` for careful smoke tests; `4` is a reasonable bounded batch default.
- `max_run_attempts`: how often a scenario may restart run-all if a normal step fails or appears stuck. Completed steps are kept; the script does not reset/undo steps.
- `max_deep_research_attempts`: separate cap for fresh Deep Research failures. Recommended default is low because DR is expensive; later normal-step retries can still reuse a successful DR result.
- `max_stuck_minutes`: retry threshold when run-all stays running without observable progress. This is not a total runtime limit. Set to `0` to disable the stuck guard; the default is deliberately generous because single LLM/DR calls can be slow.
- For long Deep Research batch tests, set `DEEP_RESEARCH_MAX_POLL_SECONDS=3900` in `.env`, restart the backend, and reset the value before production.
- `output_dir`: root folder for batch outputs. Each call to `run_batch(config)` creates a timestamped child folder unless `batch_id` is set.
- `batch_id`: leave empty by default so each fresh batch gets its own timestamped folder. Set it only when you deliberately want to append to or resume a named batch.
- `resume_completed`: `True` by default. When rerunning the same `batch_id`, scenarios already logged as `success` are skipped; failed or missing scenarios run again.
- `limit_scenarios`: optional safety cap for smoke tests. Set to `None` for the full matrix.

In [ ]:
import os
import sys
from dataclasses import replace
from pathlib import Path

repo_root = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from scripts.batch_full_run import BatchConfig, ModelSpec, failed_existing_sessions, load_batch_result, load_dotenv_if_available, preview_scenarios, quality_output_dir_for_batch, resume_existing_sessions, run_batch

load_dotenv_if_available(repo_root / ".env")

In [ ]:
# One repetition for every built-in law pair × both Gemini models × with/without DR.
# Empty law_pairs means: use all discovered built-in law pairs.
config = BatchConfig(
    base_url="http://localhost:5000",
    email=os.environ["CCC_BATCH_EMAIL"],
    password=os.environ["CCC_BATCH_PASSWORD"],
    law_pairs=(),
    models=(
        ModelSpec("gemini", "gemini-3.1-pro-preview"),
        ModelSpec("gemini", "gemini-3.5-flash"),
        ModelSpec("gemini", "gemini-3.6-flash")
    ),
    deep_research_modes=(True, False),
    repetitions=1,
    concurrency=4,
    max_run_attempts=5,
    max_deep_research_attempts=2,
    max_export_attempts=2,
    max_stuck_minutes=66,
    output_dir=repo_root / "batch_runs",
    batch_id="",
    built_in_laws_dir=repo_root / "resources" / "built_in_laws",
    limit_scenarios=None,
)

# Smoke run variant: uncomment after defining the full config if you want a safe tiny run.
# config = replace(config, law_pairs=("arbeitstagepauschale",), models=(ModelSpec("gemini", "gemini-3.5-flash"),), deep_research_modes=(False,), repetitions=1, concurrency=1, max_run_attempts=2, limit_scenarios=1, batch_id="smoke_test")

## Preview matrix

Run this before spending money. It shows the exact scenario matrix created from the active `config`.


In [ ]:
scenario_preview = preview_scenarios(config)
scenario_preview

## Run batch

Starts the configured matrix. Existing successful scenarios in the same `batch_id` are skipped when `resume_completed=True`. With the default empty `batch_id`, the runner creates a new timestamped folder and stores it in `batch_dir` for later inspection or resume.


In [ ]:
results = await run_batch(config)
batch_dir = results.output_dir
batch_dir

## Reload an existing batch

Use this after editing/backfilling `runs.jsonl`, or after reopening the notebook. Point `batch_dir` at the concrete batch folder containing `runs.jsonl`. This only loads `results`; the display tables are produced in the Results section.

In [ ]:
# Reload any existing batch by setting the concrete folder name.
# If you just ran a batch above, `batch_dir` already points to the right folder.
# batch_dir = repo_root / "batch_runs" / "batch_YYYYMMDDTHHMMSSZ"

results = load_batch_result(batch_dir)
results.output_dir

## Results

These cells only inspect the current `results` object. `mode="latest"` keeps one row per `scenario_id`; appended resume rows replace earlier failed rows in the main view, while `batch_log_total_run_attempts` keeps the accumulated run-all attempt count. `mode="all_attempts"` shows every logged row.

In [ ]:
df = results.to_dataframe(include_diagnostics=True, mode="latest")
df

In [ ]:
diagnostics = results.to_dataframe(include_diagnostics=True, mode="all_attempts")
diagnostics

In [ ]:
summary = results.to_summary_dataframe(mode="latest")
summary

In [ ]:
df.groupby(["model", "deep_research", "status"]).size().unstack(fill_value=0)

## Quality report

For a single analysed batch, use the batch folder name for the report output as well. This keeps `batch_runs/<batch_id>` and `code_analysis/output/<batch_id>` aligned.


In [ ]:
quality_output_dir = quality_output_dir_for_batch(
    batch_dir,
    output_root=repo_root / "code_analysis" / "output",
)
quality_output_dir

## Resume existing sessions

Use this when a scenario already created a useful app session and you want to continue that exact session instead of creating a fresh one. The helper appends a new audit row to the same `runs.jsonl` and `summary.csv`, while `mode="latest"` summaries still count one row per `scenario_id`.

Choose one mapping source below: either type `scenario_id -> app_session_id` manually, or derive it from the latest failed/export-failed rows in the loaded batch. You can override retry/runtime settings directly in the function call, for example `max_deep_research_attempts=3`, `max_stuck_minutes=66`, or `concurrency=1`. The helper appends to `batch_dir`, so make sure `batch_dir` points at the batch folder you want to update.


In [ ]:
# Option A: manually specify exact existing sessions.
# sessions_to_resume = {
#     "betriebsausgabenpauschale__gemini-3.6-flash__dr__01": "K3N95Y",
#     "einkunftsarten__gemini-3.5-flash__dr__01": "KMQW0R",
# }

# Option B: resume latest failed/export-failed rows from the current `results`.
sessions_to_resume = failed_existing_sessions(results)
sessions_to_resume

In [ ]:
resume_config = replace(config, batch_id=batch_dir.name)

resume_results = await resume_existing_sessions(
    resume_config,
    sessions_to_resume,
    concurrency=1,
    max_deep_research_attempts=3,
    max_export_attempts=3,
    max_stuck_minutes=66,
)

# Reload the full batch including the appended resume rows, then rerun the Results cells.
results = load_batch_result(batch_dir)
resume_results.to_dataframe(include_diagnostics=True, mode="all_attempts")